# VidClips — Windows build on demand

Use this notebook only when you want a new Windows installer. Google Colab runs Linux, so it triggers the repository's **Windows GitHub Actions runner**; it does not try to compile a Windows `.exe` inside Linux.

## Before running
1. Create a GitHub token with **Actions: Read and write** and **Contents: Read-only** access to `vidclipsltd/vidclips`.
2. Run the cells in order and paste the token only into the hidden prompt. Do not write the token into this notebook or commit it to GitHub.
3. The notebook triggers a build from `main`, waits for completion, then downloads and extracts the installer artifact.

The Windows workflow is manual-only; pushing code does not automatically create installers.

In [ ]:
import getpass, time, os, zipfile
from datetime import datetime, timezone
import requests

REPO = 'vidclipsltd/vidclips'
WORKFLOW = 'desktop-windows.yml'
TOKEN = getpass.getpass('GitHub token (hidden input): ').strip()
if not TOKEN:
    raise ValueError('A GitHub token is required to start a build.')
API = 'https://api.github.com'
HEADERS = {'Authorization': f'Bearer {TOKEN}', 'Accept': 'application/vnd.github+json', 'X-GitHub-Api-Version': '2022-11-28'}
check = requests.get(f'{API}/repos/{REPO}', headers=HEADERS, timeout=30)
check.raise_for_status()
print('Connected to:', check.json()['full_name'])
print('Token is held only in this Colab runtime. Do not share the notebook with token output.')

In [ ]:
# Trigger a Windows installer build from main (only when this cell is run).
requested_at = datetime.now(timezone.utc)
dispatch_url = f'{API}/repos/{REPO}/actions/workflows/{WORKFLOW}/dispatches'
response = requests.post(dispatch_url, headers=HEADERS, json={'ref': 'main'}, timeout=30)
if response.status_code not in (204, 201):
    raise RuntimeError(f'Could not trigger build ({response.status_code}): {response.text}')
print('Build requested for main at', requested_at.isoformat())
print('Waiting for GitHub Actions to register the run...')

run = None
runs_url = f'{API}/repos/{REPO}/actions/workflows/{WORKFLOW}/runs?event=workflow_dispatch&per_page=10'
for attempt in range(30):
    data = requests.get(runs_url, headers=HEADERS, timeout=30)
    data.raise_for_status()
    candidates = [r for r in data.json().get('workflow_runs', [])
                  if r.get('event') == 'workflow_dispatch'
                  and r.get('head_branch') == 'main'
                  and datetime.fromisoformat(r['created_at'].replace('Z', '+00:00')) >= requested_at]
    if candidates:
        run = max(candidates, key=lambda r: r['created_at'])
        break
    time.sleep(5)
if run is None:
    raise TimeoutError('GitHub has not registered the build yet. Re-run this cell only after checking the Actions page to avoid starting duplicate builds.')
run_id = run['id']
print('Build run:', run_id, run['html_url'])

while True:
    status_response = requests.get(f'{API}/repos/{REPO}/actions/runs/{run_id}', headers=HEADERS, timeout=30)
    status_response.raise_for_status()
    status = status_response.json()
    print('Status:', status['status'], '| conclusion:', status.get('conclusion'))
    if status['status'] == 'completed':
        break
    time.sleep(15)
if status.get('conclusion') != 'success':
    raise RuntimeError(f"Windows build did not succeed ({status.get('conclusion')}). Open the run logs: {status['html_url']}")
print('Windows build succeeded.')

In [ ]:
# Download the installer artifact from the completed run.
artifact_response = requests.get(f'{API}/repos/{REPO}/actions/runs/{run_id}/artifacts', headers=HEADERS, timeout=30)
artifact_response.raise_for_status()
artifacts = [a for a in artifact_response.json().get('artifacts', []) if a['name'] == 'VidClips-Windows-Installer' and not a.get('expired')]
if not artifacts:
    raise FileNotFoundError('The Windows installer artifact was not found. Check the build run logs.')
artifact = max(artifacts, key=lambda a: a['created_at'])
zip_response = requests.get(artifact['archive_download_url'], headers=HEADERS, timeout=180)
zip_response.raise_for_status()
zip_path = '/content/VidClips-Windows-Installer.zip'
with open(zip_path, 'wb') as f:
    f.write(zip_response.content)
output_dir = '/content/VidClips-Windows-Installer'
os.makedirs(output_dir, exist_ok=True)
with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(output_dir)
installers = [os.path.join(root, name) for root, _, names in os.walk(output_dir) for name in names if name.lower().endswith('.exe')]
print('Installer artifact saved to:', zip_path)
for installer in installers:
    print('Windows installer:', installer)
if not installers:
    print('No .exe was found in the artifact. Check the artifact contents and workflow logs.')
print('Build run:', status['html_url'])

## Notes
- Download the `.exe` to your Windows PC before the Colab runtime expires.
- If you need another build, run the dispatch cell again only after the previous run is finished.
- The installer still relies on the existing local Python AI setup described in `desktop/README.md`; this notebook does not package the entire Python AI environment.